# Material Candidate Explorer V11 — Prompt-Guided Real MatterGen Generation

This notebook is based directly on the previously successful **MatterGen T4 High-Yield V10** execution path.

It does not stop after generating a search plan.

Execution order:

1. Accept a free-form materials-discovery prompt.
2. Retrieve current literature with the repository's grounded RAG code.
3. Convert the evidence into executable MatterGen chemical systems.
4. Load MatterGen once and generate multiple real CIF structures.
5. Evaluate every retained candidate with MatterSim and CHGNet.
6. Preserve candidates from every chemical system and control profile without filename collisions.
7. Produce a ranked table of multiple candidates with raw predictions, provenance, model disagreement, and warnings.
8. Package all CIFs, evidence, logs, and tables into one ZIP and download it.

The validated V10 MatterGen installation and generation route is retained. The earlier replacement installer is not used.

Select **Runtime → Change runtime type → T4 GPU**, then use **Runtime → Run all**.


In [ ]:
#@title 0. Discovery request and run settings
DISCOVERY_PROMPT = "" #@param {type:"string"}
MODE = "PROMPT_GUIDED_CRYSTAL" #@param ["PROMPT_GUIDED_CRYSTAL", "DIRECT_CHEMICAL_SYSTEMS"]
GENOMIC_MODE = "OFF" #@param ["OFF", "GENOMIC_VARIANT_DISCOVERY", "REGULATORY_SEQUENCE_DESIGN", "SPLICE_VARIANT_ANALYSIS", "DISEASE_VARIANT_PRIORITIZATION", "ALPHAGENOME_EVALUATION_ONLY"]
GENOMIC_CANDIDATES_JSON = "" #@param {type:"string"}
DIRECT_CHEMICAL_SYSTEMS = "" #@param {type:"string"}
QUALITY = "MAXIMUM" #@param ["QUICK", "STANDARD", "DEEP", "MAXIMUM"]

RAG_MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507" #@param {type:"string"}
FROM_DATE = "2024-01-01" #@param {type:"date"}
CONTACT_EMAIL = "" #@param {type:"string"}
NCBI_API_KEY = "" #@param {type:"string"}
OPENALEX_API_KEY = "" #@param {type:"string"}

MAX_HOURS = 5.0 #@param {type:"slider", min:1, max:12, step:0.5}
FINAL_RESERVE_MINUTES = 12

RUN_LITERATURE_RAG = True
RUN_DIRECT_MATTERSIM = True
RUN_OFFICIAL_MATTERGEN_EVALUATION = True
USE_MATTERSIM_5M = True
RUN_CHGNET_CROSSCHECK = True

MOUNT_DRIVE = False
BROWSER_DOWNLOAD_FINAL = True
BASE_SEED = 20260715

# The successful V10 route is retained.
BATCH_SIZE_CANDIDATES = [4, 2, 1]

QUALITY_CONFIG = {
    "QUICK": {
        "max_systems": 1,
        "samples_per_profile": 2,
        "rag_results": 15,
        "rag_branches": 12,
    },
    "STANDARD": {
        "max_systems": 2,
        "samples_per_profile": 4,
        "rag_results": 25,
        "rag_branches": 20,
    },
    "DEEP": {
        "max_systems": 3,
        "samples_per_profile": 4,
        "rag_results": 40,
        "rag_branches": 32,
    },
    "MAXIMUM": {
        "max_systems": 4,
        "samples_per_profile": 4,
        "rag_results": 60,
        "rag_branches": 48,
    },
}

if QUALITY not in QUALITY_CONFIG:
    raise ValueError(f"Unsupported QUALITY: {QUALITY}")

print("Mode:", MODE)
print("Quality:", QUALITY)
print("The Hugging Face token will be requested securely in the next cell.")


In [ ]:
#@title 1. Clone the repository, retrieve evidence, and resolve executable MatterGen systems
from __future__ import annotations

import getpass
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import date
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU runtime is required. Select Runtime > Change runtime type > T4 GPU."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", GPU_NAME, f"{GPU_GB:.2f} GiB")

prompt = DISCOVERY_PROMPT.strip()
if not prompt:
    prompt = input("Discovery prompt: ").strip()
if not prompt:
    raise ValueError("The discovery prompt cannot be empty.")
DISCOVERY_PROMPT = prompt

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Hugging Face token (required; hidden input): "
    ).strip()
if not HF_TOKEN:
    raise ValueError("A Hugging Face token is required.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["NCBI_API_KEY"] = NCBI_API_KEY.strip()
os.environ["OPENALEX_API_KEY"] = OPENALEX_API_KEY.strip()
os.environ["LITERATURE_CONTACT_EMAIL"] = CONTACT_EMAIL.strip()
os.environ["RAG_MODEL_API_URL"] = "https://router.huggingface.co/v1"
os.environ["RAG_MODEL_NAME"] = RAG_MODEL.strip()
os.environ["RAG_MODEL_API_KEY"] = HF_TOKEN
os.environ["RAG_MODEL_TIMEOUT_SECONDS"] = "300"

REPOSITORY = "https://github.com/jujumelona/material-candidate-explorer.git"
REPO_DIR = Path("/content/material-candidate-explorer")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)],
    check=True,
)

PROJECT_ROOT = (
    REPO_DIR
    if (REPO_DIR / "pyproject.toml").exists()
    else REPO_DIR / "SCIENTIFIC_DISCOVERY_GENERATOR"
)
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"The cloned repository does not contain the project package: {PROJECT_ROOT}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openai>=1.0",
        "-e",
        str(PROJECT_ROOT),
    ],
    check=True,
)

manifest_path = PROJECT_ROOT / "integrations" / "manifest.v1.json"
PROJECT_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
COMPONENTS = {
    item["component_id"]: item for item in PROJECT_MANIFEST.get("components", [])
}
for required in ("mattergen", "mattersim", "chgnet"):
    if required not in COMPONENTS:
        raise RuntimeError(f"Repository manifest is missing {required!r}")

MG = COMPONENTS["mattergen"]
MS = COMPONENTS["mattersim"]
CHG = COMPONENTS["chgnet"]

repo_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()
PROJECT_ZIP_SHA256 = f"github:{repo_commit}"

REQUEST_ARTIFACTS = Path("/content/MCE_PROMPT_ARTIFACTS")
REQUEST_ARTIFACTS.mkdir(parents=True, exist_ok=True)
RAG_BUNDLE_PATH = REQUEST_ARTIFACTS / "literature_evidence_bundle.json"
PROMPT_RESOLUTION_PATH = REQUEST_ARTIFACTS / "prompt_resolution.json"

cfg = QUALITY_CONFIG[QUALITY]


def valid_system(value: str) -> bool:
    return bool(re.fullmatch(r"[A-Z][a-z]?(?:-[A-Z][a-z]?)+", value.strip()))


def unique_systems(values, limit: int) -> list[str]:
    output = []
    for item in values:
        value = str(item).strip()
        if valid_system(value) and value not in output:
            output.append(value)
    return output[:limit]


def call_hf_json(system_prompt: str, user_payload: dict) -> dict:
    from openai import OpenAI

    client = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=HF_TOKEN,
        timeout=300,
    )
    kwargs = {
        "model": RAG_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": json.dumps(user_payload, ensure_ascii=False),
            },
        ],
        "temperature": 0.1,
        "max_tokens": 3500,
    }
    try:
        result = client.chat.completions.create(
            **kwargs,
            response_format={"type": "json_object"},
        )
    except Exception:
        result = client.chat.completions.create(**kwargs)

    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("The Hugging Face model returned an empty response.")
    match = re.search(r"\{.*\}", content, re.S)
    if not match:
        raise RuntimeError("The Hugging Face model did not return a JSON object.")
    value = json.loads(match.group(0))
    if not isinstance(value, dict):
        raise RuntimeError("The Hugging Face response was not a JSON object.")
    return value


rag_bundle_json = None
rag_error = None

if MODE == "PROMPT_GUIDED_CRYSTAL" and RUN_LITERATURE_RAG:
    try:
        from discovery_os.literature_rag import (
            JsonEvidenceIndex,
            LiteratureSource,
            build_literature_rag_from_environment,
            save_evidence_bundle,
        )

        pipeline = build_literature_rag_from_environment(require_model=True)
        evidence_index = JsonEvidenceIndex(
            REQUEST_ARTIFACTS / "literature_index"
        )
        bundle = pipeline.run(
            DISCOVERY_PROMPT,
            sources=[
                LiteratureSource.PUBMED,
                LiteratureSource.EUROPE_PMC,
                LiteratureSource.OPENALEX,
                LiteratureSource.CROSSREF,
                LiteratureSource.ARXIV,
            ],
            from_date=date.fromisoformat(FROM_DATE),
            max_results_per_query=int(cfg["rag_results"]),
            max_branches=int(cfg["rag_branches"]),
            index=evidence_index,
        )
        save_evidence_bundle(bundle, RAG_BUNDLE_PATH)
        rag_bundle_json = bundle.model_dump(mode="json")
        print(
            "Literature retrieval completed:",
            len(bundle.records),
            "records,",
            len(bundle.claims),
            "grounded claims,",
            len(bundle.branches),
            "search branches.",
        )
    except Exception as exc:
        rag_error = f"{type(exc).__name__}: {exc}"
        print("Literature retrieval warning:", rag_error)
        print(
            "Generation will continue using the prompt model rather than stopping."
        )

if MODE == "DIRECT_CHEMICAL_SYSTEMS":
    parsed = re.split(r"[,;\s]+", DIRECT_CHEMICAL_SYSTEMS.strip())
    CHEMICAL_SYSTEMS = unique_systems(parsed, int(cfg["max_systems"]))
    if not CHEMICAL_SYSTEMS:
        raise ValueError(
            "DIRECT_CHEMICAL_SYSTEMS must contain values such as Mg-B, La-H, or Li-O."
        )
    PROMPT_RESOLUTION = {
        "mode": MODE,
        "chemical_systems": CHEMICAL_SYSTEMS,
        "conditions": {"energy_above_hull": 0.0},
        "rationale": "Direct chemical-system input supplied by the user.",
        "unsupported_targets": [],
        "literature_rag_error": rag_error,
    }
else:
    compact_evidence = {}
    if rag_bundle_json:
        compact_evidence = {
            "claims": rag_bundle_json.get("claims", [])[:80],
            "branches": rag_bundle_json.get("branches", [])[:60],
            "records": rag_bundle_json.get("records", [])[:40],
        }

    resolver_system = f"""You convert a materials-discovery request and grounded literature evidence into executable MatterGen crystal-generation conditions.

This JSON is immediately consumed by MatterGen. It is not the final result and must not merely describe a plan.

Return 1 to {int(cfg['max_systems'])} chemical systems. Each system must contain only unique element symbols joined by hyphens, for example Mg-B, La-H, Fe-As, or Cu-O. Do not return a compound formula such as MgB2.

MatterGen supports these conditions:
energy_above_hull, dft_band_gap, dft_mag_density, ml_bulk_modulus, hhi_score, space_group.

Always include energy_above_hull = 0.0.
Only add other supported conditions when directly justified by the request or evidence.
Do not invent superconducting transition temperature, toxicity, synthesis success, or any unsupported value.

Return JSON keys:
chemical_systems, conditions, rationale, unsupported_targets, evidence_branch_ids.
"""
    PROMPT_RESOLUTION = call_hf_json(
        resolver_system,
        {
            "user_prompt": DISCOVERY_PROMPT,
            "grounded_evidence": compact_evidence,
        },
    )
    CHEMICAL_SYSTEMS = unique_systems(
        PROMPT_RESOLUTION.get("chemical_systems", []),
        int(cfg["max_systems"]),
    )
    if not CHEMICAL_SYSTEMS:
        raise RuntimeError(
            "The prompt resolver did not return any valid MatterGen chemical systems."
        )

    raw_conditions = (
        PROMPT_RESOLUTION.get("conditions")
        if isinstance(PROMPT_RESOLUTION.get("conditions"), dict)
        else {}
    )
    supported = {
        "energy_above_hull",
        "dft_band_gap",
        "dft_mag_density",
        "ml_bulk_modulus",
        "hhi_score",
        "space_group",
    }
    conditions = {"energy_above_hull": 0.0}
    for key, value in raw_conditions.items():
        if key in supported and key != "energy_above_hull" and value is not None:
            conditions[key] = value
    PROMPT_RESOLUTION["conditions"] = conditions
    PROMPT_RESOLUTION["chemical_systems"] = CHEMICAL_SYSTEMS
    PROMPT_RESOLUTION["literature_rag_error"] = rag_error

PROMPT_RESOLUTION_PATH.write_text(
    json.dumps(PROMPT_RESOLUTION, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

samples = int(cfg["samples_per_profile"])
base_profiles = [
    {
        "name": "stable",
        "energy_above_hull": 0.00,
        "guidance": 2.0,
        "samples": samples,
    },
    {
        "name": "balanced",
        "energy_above_hull": 0.05,
        "guidance": 2.0,
        "samples": samples,
    },
    {
        "name": "diverse",
        "energy_above_hull": 0.05,
        "guidance": 1.5,
        "samples": samples,
    },
]

GENERATION_PROFILES = []
for system_index, chemical_system in enumerate(CHEMICAL_SYSTEMS):
    safe_system = chemical_system.replace("-", "_")
    for profile in base_profiles:
        item = dict(profile)
        item["chemical_system"] = chemical_system
        item["name"] = f"{safe_system}_{profile['name']}"
        item["system_index"] = system_index
        GENERATION_PROFILES.append(item)

# Compatibility only. Generation uses every item in CHEMICAL_SYSTEMS.
CHEMICAL_SYSTEM = CHEMICAL_SYSTEMS[0]

print("Executable chemical systems:", CHEMICAL_SYSTEMS)
print("Generation profiles:", len(GENERATION_PROFILES))
print(
    "Total requested real structures:",
    sum(int(item["samples"]) for item in GENERATION_PROFILES),
)
print("Repository commit:", repo_commit)


In [ ]:
# ===== 단일 실행: 설치 → 즉시 MatterGen 생성 → 평가 → 무조건 ZIP =====
import contextlib
import csv
import gc
import json
import math
import os
import re
import hashlib
import selectors
import shutil
import signal
import subprocess
import sys
import time
import traceback
import zipfile
from pathlib import Path
from typing import Any

RUN_START = time.time()
HARD_END = RUN_START + float(MAX_HOURS) * 3600
FINAL_RESERVE_SECONDS = int(FINAL_RESERVE_MINUTES * 60)

WORK = Path("/content/direct_real_models_v11_prompt_rag")
SOURCE = WORK / "source"
ENVS = WORK / "envs"
CACHE = WORK / "cache"
RESULTS = WORK / "results"
LOGS = WORK / "logs"
RAW = RESULTS / "raw"
CIFS = RESULTS / "cifs"
GENERATION_DIR = RESULTS / "mattergen_generation"
for p in (WORK, SOURCE, ENVS, CACHE, RESULTS, LOGS, RAW, CIFS, GENERATION_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Preserve the prompt, grounded literature evidence, and executable resolution.
write_prompt_artifacts_later = True

os.environ.update({
    "HF_HOME": str(CACHE / "huggingface"),
    "HUGGINGFACE_HUB_CACHE": str(CACHE / "huggingface" / "hub"),
    "TORCH_HOME": str(CACHE / "torch"),
    "UV_CACHE_DIR": str(CACHE / "uv"),
    "UV_HTTP_TIMEOUT": "600",
    "UV_NO_PROGRESS": "1",
    "UV_INDEX_STRATEGY": "unsafe-best-match",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONNOUSERSITE": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
    "HF_HUB_DOWNLOAD_TIMEOUT": "600",
    "HF_HUB_ETAG_TIMEOUT": "60",
    "HF_HUB_DISABLE_XET": "1",
})
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()

DRIVE_OUT = None
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_OUT = Path("/content/drive/MyDrive/SCIENTIFIC_DISCOVERY_OUTPUT/DIRECT_REAL_MODEL_RUN_V10")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def remaining_seconds() -> int:
    return max(0, int(HARD_END - time.time()))

def free_gb() -> float:
    return shutil.disk_usage("/content").free / 1024**3

def gpu_status() -> str:
    try:
        p = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.free,utilization.gpu", "--format=csv,noheader,nounits"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, timeout=10,
        )
        return p.stdout.strip() or "unavailable"
    except Exception:
        return "unavailable"

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def allowed_timeout(requested: int) -> int:
    allowed = remaining_seconds() - FINAL_RESERVE_SECONDS
    if allowed < 60:
        raise TimeoutError("최종 결과 ZIP 저장 시간을 제외하면 실행 시간이 부족합니다.")
    return max(60, min(int(requested), int(allowed)))

def kill_group(proc: subprocess.Popen) -> None:
    if proc.poll() is not None:
        return
    with contextlib.suppress(Exception):
        os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=15)
    except Exception:
        with contextlib.suppress(Exception):
            os.killpg(proc.pid, signal.SIGKILL)

def run_logged(command, *, name, cwd=None, timeout=3600, retries=1, check=True, retry_wait=15, env_overrides=None):
    command = [str(x) for x in command]
    log_path = LOGS / f"{name}.log"
    final_rc = 999
    final_tail = ""
    for attempt in range(1, retries + 1):
        effective = allowed_timeout(timeout)
        started = time.time()
        print(f"\n[{name}] {attempt}/{retries}")
        print("$", " ".join(command))
        print(f"timeout={effective}s disk={free_gb():.1f}GB gpu={gpu_status()}")
        with log_path.open("a", encoding="utf-8") as log:
            log.write(f"\n===== ATTEMPT {attempt}/{retries} =====\n$ {' '.join(command)}\n")
            child_env = os.environ.copy()
            if env_overrides:
                for key, value in env_overrides.items():
                    if value is None:
                        child_env.pop(str(key), None)
                    else:
                        child_env[str(key)] = str(value)
            proc = subprocess.Popen(
                command,
                cwd=str(cwd) if cwd else None,
                env=child_env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=False,
                bufsize=0,
                start_new_session=True,
            )
            assert proc.stdout is not None
            selector = selectors.DefaultSelector()
            selector.register(proc.stdout, selectors.EVENT_READ)
            last_beat = time.time()
            tail = bytearray()
            try:
                while True:
                    if time.time() - started > effective:
                        raise subprocess.TimeoutExpired(command, effective)
                    events = selector.select(timeout=1.0)
                    for key, _ in events:
                        chunk = os.read(key.fileobj.fileno(), 65536)
                        if chunk:
                            text = chunk.decode("utf-8", errors="replace")
                            print(text, end="", flush=True)
                            log.write(text)
                            log.flush()
                            tail.extend(chunk)
                            if len(tail) > 250000:
                                del tail[:-200000]
                    if time.time() - last_beat >= 30:
                        beat = f"\n[{name}] running {int(time.time()-started)}s | disk {free_gb():.1f}GB | gpu {gpu_status()}\n"
                        print(beat, end="", flush=True)
                        log.write(beat); log.flush()
                        last_beat = time.time()
                    if proc.poll() is not None:
                        rest = proc.stdout.read()
                        if rest:
                            text = rest.decode("utf-8", errors="replace")
                            print(text, end="", flush=True)
                            log.write(text); log.flush()
                            tail.extend(rest)
                        break
                final_rc = int(proc.returncode or 0)
            except BaseException as exc:
                kill_group(proc)
                log.write(f"\nPROCESS_EXCEPTION: {type(exc).__name__}: {exc}\n")
                log.flush()
                final_rc = -9
            finally:
                with contextlib.suppress(Exception):
                    selector.close()
            final_tail = bytes(tail[-200000:]).decode("utf-8", errors="replace")
        if final_rc == 0:
            return subprocess.CompletedProcess(command, 0, stdout=final_tail)
        print(f"[{name}] 실패 returncode={final_rc}")
        if attempt < retries and remaining_seconds() > FINAL_RESERVE_SECONDS + retry_wait + 60:
            time.sleep(retry_wait)
    result = subprocess.CompletedProcess(command, final_rc, stdout=final_tail)
    if check:
        raise RuntimeError(f"{name} 실패(returncode={final_rc}). 로그: {log_path}")
    return result

def collect_cifs() -> list[Path]:
    # MatterGen writes identically named CIF archives in every profile directory.
    # Extract each archive inside its own parent directory to prevent overwrites.
    for archive in GENERATION_DIR.rglob("*.zip"):
        if "cif" not in archive.name.lower():
            continue
        target = archive.parent / f"extracted_{archive.stem}"
        shutil.rmtree(target, ignore_errors=True)
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            for info in zf.infolist():
                member = Path(info.filename)
                if info.is_dir() or member.suffix.lower() != ".cif":
                    continue
                if member.is_absolute() or ".." in member.parts:
                    raise RuntimeError(f"Unsafe CIF path: {info.filename}")
                (target / member.name).write_bytes(zf.read(info))

    for old in CIFS.glob("*.cif"):
        old.unlink()

    profile_by_name = {
        str(item["name"]): item for item in GENERATION_PROFILES
    }
    seen_hashes = set()
    provenance = []
    retained = []

    candidates = sorted(GENERATION_DIR.rglob("*.cif"))
    for source in candidates:
        digest = hashlib.sha256(source.read_bytes()).hexdigest()
        if digest in seen_hashes:
            continue
        seen_hashes.add(digest)

        relative = source.relative_to(GENERATION_DIR)
        top_dir = relative.parts[0] if relative.parts else ""
        profile_name = top_dir.split("_", 1)[1] if "_" in top_dir else top_dir
        profile = profile_by_name.get(profile_name, {})
        safe_profile = re.sub(r"[^A-Za-z0-9_.-]+", "_", profile_name)
        destination = CIFS / (
            f"candidate_{len(retained):04d}__{safe_profile}__{digest[:10]}.cif"
        )
        shutil.copy2(source, destination)
        retained.append(destination)
        provenance.append(
            {
                "candidate": destination.name,
                "source_relative_path": str(relative),
                "sha256": digest,
                "profile": profile_name,
                "chemical_system": profile.get("chemical_system"),
                "energy_above_hull_condition_eV_per_atom": profile.get(
                    "energy_above_hull"
                ),
                "guidance_factor": profile.get("guidance"),
                "requested_profile_samples": profile.get("samples"),
            }
        )

    write_json(RAW / "candidate_provenance.json", provenance)
    return retained

STATUS = {
    "status": "running",
    "gpu": GPU_NAME,
    "project_zip_sha256": PROJECT_ZIP_SHA256,
    "manifest_revision": PROJECT_MANIFEST.get("manifest_revision"),
    "mattergen": {"success": False},
    "official_evaluation": {
        "requested": bool(RUN_OFFICIAL_MATTERGEN_EVALUATION),
        "success": False,
    },
    "mattersim": {
        "requested": bool(RUN_DIRECT_MATTERSIM),
        "success": False,
    },
    "chgnet": {
        "requested": bool(RUN_CHGNET_CROSSCHECK),
        "success": False,
    },
}

def package_all(status: str, error: str | None = None) -> Path:
    STATUS["status"] = status
    STATUS["error"] = error
    STATUS["elapsed_seconds"] = time.time() - RUN_START
    STATUS["remaining_seconds"] = remaining_seconds()
    STATUS["free_disk_gb"] = free_gb()
    STATUS["gpu_final"] = gpu_status()
    write_json(RAW / "project_manifest_snapshot.json", PROJECT_MANIFEST)
    write_json(RESULTS / "final_report.json", STATUS)
    final_zip = Path("/content/MATERIAL_CANDIDATE_EXPLORER_V11_RESULTS.zip")
    final_zip.unlink(missing_ok=True)
    with zipfile.ZipFile(final_zip, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for root in (RESULTS, LOGS):
            for path in root.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(WORK))
    if DRIVE_OUT is not None:
        shutil.copy2(final_zip, DRIVE_OUT / final_zip.name)
    return final_zip

def install_eval_env(env_name: str, package: str, version: str) -> Path:
    env = ENVS / env_name
    py = env / "bin/python"
    if not py.exists():
        run_logged([UV, "venv", str(env), "--python", "3.12"], name=f"{env_name}_create", timeout=600, retries=2)
        run_logged(
            [UV, "pip", "install", "--python", str(py), f"{package}=={version}", "setuptools<81"],
            name=f"{env_name}_install", timeout=3600, retries=2,
        )
    return py

# Store prompt and RAG evidence in the same result tree.
write_json(RAW / "prompt_resolution.json", PROMPT_RESOLUTION)
write_json(
    RAW / "discovery_request.json",
    {
        "prompt": DISCOVERY_PROMPT,
        "mode": MODE,
        "quality": QUALITY,
        "chemical_systems": CHEMICAL_SYSTEMS,
        "rag_model": RAG_MODEL,
        "from_date": FROM_DATE,
        "repository_commit": PROJECT_ZIP_SHA256,
    },
)
if RAG_BUNDLE_PATH.exists():
    shutil.copy2(RAG_BUNDLE_PATH, RAW / RAG_BUNDLE_PATH.name)
index_root = REQUEST_ARTIFACTS / "literature_index"
if index_root.exists():
    shutil.copytree(
        index_root,
        RAW / "literature_index",
        dirs_exist_ok=True,
    )

FINAL_ZIP = None
fatal_error = None
try:
    uv_version = str(PROJECT_MANIFEST.get("uv_version") or "0.11.28")
    run_logged(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", f"uv=={uv_version}"],
        name="install_uv", timeout=900, retries=3,
    )
    UV = shutil.which("uv")
    if not UV:
        raise FileNotFoundError("uv 실행 파일을 찾지 못했습니다.")
    run_logged([UV, "python", "install", "3.10"], name="install_python310", timeout=1200, retries=3)
    if RUN_CHGNET_CROSSCHECK:
        run_logged([UV, "python", "install", "3.12"], name="install_python312", timeout=1200, retries=3)

    mg_source = SOURCE / "mattergen"
    mg_revision = MG["source"]["revision"]
    if not (mg_source / ".git").exists():
        shutil.rmtree(mg_source, ignore_errors=True)
        mg_source.mkdir(parents=True, exist_ok=True)
        run_logged(["git", "init", "-q"], name="mattergen_git_init", cwd=mg_source, timeout=120)
        run_logged(["git", "remote", "add", "origin", MG["source"]["repository"]], name="mattergen_git_remote", cwd=mg_source, timeout=120)
        run_logged(
            ["git", "-c", "http.version=HTTP/1.1", "fetch", "--depth", "1", "origin", mg_revision],
            name="mattergen_git_fetch", cwd=mg_source, timeout=1800, retries=3,
        )
        run_logged(["git", "checkout", "--detach", "FETCH_HEAD"], name="mattergen_git_checkout", cwd=mg_source, timeout=300)

    mg_env = ENVS / "mattergen_py310"
    mg_py = mg_env / "bin/python"
    mg_eval_cli = mg_env / "bin/mattergen-evaluate"
    constraints = WORK / "mattergen_constraints.txt"
    constraints.write_text("\n".join([
        "mattersim==1.1.2",
        "numpy==1.26.4",
        "setuptools==80.9.0",
        "pytorch-lightning==2.0.6",
    ]) + "\n", encoding="utf-8")

    if not mg_py.exists():
        shutil.rmtree(mg_env, ignore_errors=True)
        run_logged([UV, "venv", str(mg_env), "--python", "3.10"], name="mattergen_create_env", timeout=600, retries=2)
        install_cmd = [
            UV, "pip", "install", "--python", str(mg_py),
            "--exclude-newer", "2025-07-24T23:59:59Z",
            "--extra-index-url", "https://download.pytorch.org/whl/cu118",
            "--find-links", "https://data.pyg.org/whl/torch-2.2.0+cu118.html",
            "--constraint", str(constraints),
            "--editable", str(mg_source),
        ]
        run_logged(install_cmd, name="mattergen_official_source_install", cwd=mg_source, timeout=7200, retries=2)
        run_logged(
            [UV, "pip", "install", "--python", str(mg_py), "--no-deps", "setuptools==80.9.0"],
            name="mattergen_lock_setuptools", timeout=900, retries=2,
        )

    shutil.rmtree(GENERATION_DIR, ignore_errors=True)
    GENERATION_DIR.mkdir(parents=True, exist_ok=True)
    profile_config = {
        "chemical_systems": list(CHEMICAL_SYSTEMS),
        "profiles": GENERATION_PROFILES,
        "batch_sizes": [int(x) for x in BATCH_SIZE_CANDIDATES],
        "base_seed": int(BASE_SEED),
        "output_root": str(GENERATION_DIR),
    }
    profile_config_path = RAW / "generation_profiles_request.json"
    write_json(profile_config_path, profile_config)

    generation_script = WORK / "run_mattergen_profiles.py"
    generation_script.write_text(r'''
import gc
import json
import os
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import torch

from mattergen.common.utils.data_classes import MatterGenCheckpointInfo
from mattergen.generator import CrystalGenerator

config_path = Path(sys.argv[1])
manifest_path = Path(sys.argv[2])
cfg = json.loads(config_path.read_text(encoding="utf-8"))
output_root = Path(cfg["output_root"])
output_root.mkdir(parents=True, exist_ok=True)

def seed_all(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

mask_override = (
    "++lightning_module.diffusion_module.model.element_mask_func="
    "{_target_:'mattergen.denoiser.mask_disallowed_elements',_partial_:True}"
)
checkpoint = MatterGenCheckpointInfo.from_hf_hub(
    "chemical_system_energy_above_hull",
    config_overrides=[mask_override],
)
generator = CrystalGenerator(
    checkpoint_info=checkpoint,
    properties_to_condition_on={},
    batch_size=1,
    num_batches=1,
    record_trajectories=False,
    diffusion_guidance_factor=0.0,
)
generator.prepare()
if torch.cuda.is_available():
    generator.model.to("cuda")

records = []
for profile_index, profile in enumerate(cfg["profiles"]):
    name = str(profile["name"])
    samples = int(profile["samples"])
    profile_dir = output_root / f"{profile_index:02d}_{name}"
    properties = {
        "chemical_system": str(profile["chemical_system"]),
        "energy_above_hull": float(profile["energy_above_hull"]),
    }
    seed = int(cfg["base_seed"]) + profile_index * 100_003
    success = False
    attempts = []

    for batch_size in cfg["batch_sizes"]:
        batch_size = int(batch_size)
        if batch_size <= 0 or samples % batch_size != 0:
            continue
        num_batches = samples // batch_size
        shutil.rmtree(profile_dir, ignore_errors=True)
        profile_dir.mkdir(parents=True, exist_ok=True)
        seed_all(seed)
        generator.properties_to_condition_on = properties
        generator.diffusion_guidance_factor = float(profile["guidance"])
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        try:
            structures = generator.generate(
                batch_size=batch_size,
                num_batches=num_batches,
                output_dir=str(profile_dir),
            )
            if len(structures) != samples:
                raise RuntimeError(f"generated {len(structures)} structures, expected {samples}")
            attempts.append({
                "batch_size": batch_size,
                "num_batches": num_batches,
                "success": True,
            })
            records.append({
                "profile": name,
                "properties": properties,
                "guidance": float(profile["guidance"]),
                "seed": seed,
                "batch_size": batch_size,
                "num_batches": num_batches,
                "generated_count": len(structures),
                "output_dir": str(profile_dir),
                "attempts": attempts,
            })
            success = True
            break
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            attempts.append({
                "batch_size": batch_size,
                "num_batches": num_batches,
                "success": False,
                "error": message,
            })
            lower = message.lower()
            is_oom = (
                "out of memory" in lower
                or "cuda error" in lower
                or "cublas" in lower
                or "cudnn" in lower
            )
            print(f"[{name}] batch={batch_size} failed: {message}", flush=True)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if not is_oom:
                raise

    if not success:
        raise RuntimeError(f"profile {name!r} failed for every allowed batch size: {attempts}")

manifest_path.write_text(
    json.dumps(
        {
            "model": "chemical_system_energy_above_hull",
            "chemical_systems": cfg["chemical_systems"],
            "total_generated": sum(x["generated_count"] for x in records),
            "profiles": records,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(manifest_path.read_text(encoding="utf-8"))
''', encoding="utf-8")

    generation_manifest = RAW / "generation_profiles_result.json"
    gc.collect()
    torch.cuda.empty_cache()
    gen = run_logged(
        [str(mg_py), str(generation_script), str(profile_config_path), str(generation_manifest)],
        name="mattergen_high_yield_generation",
        cwd=mg_source,
        timeout=14400,
        retries=1,
        check=False,
    )
    if gen.returncode != 0:
        run_logged(
            [UV, "pip", "install", "--python", str(mg_py), "--force-reinstall", "--no-deps",
             "setuptools==80.9.0", "numpy==1.26.4"],
            name="mattergen_runtime_repair", timeout=1200, retries=2,
        )
        gen = run_logged(
            [str(mg_py), str(generation_script), str(profile_config_path), str(generation_manifest)],
            name="mattergen_high_yield_generation_after_repair",
            cwd=mg_source,
            timeout=14400,
            retries=1,
            check=False,
        )
    if gen.returncode != 0:
        raise RuntimeError("MatterGen 고수율 생성이 실패했습니다. 전체 로그를 결과 ZIP에 저장했습니다.")

    cifs = collect_cifs()
    if not cifs:
        raise RuntimeError("MatterGen 생성 프로세스는 끝났지만 CIF가 없습니다.")
    generation_info = json.loads(generation_manifest.read_text(encoding="utf-8"))
    STATUS["mattergen"] = {
        "success": True,
        "version": MG["install"]["version"],
        "source_revision": mg_revision,
        "model": "chemical_system_energy_above_hull",
        "generated_cif_count": len(cifs),
        "requested_total": sum(int(x["samples"]) for x in GENERATION_PROFILES),
        "chemical_systems": list(CHEMICAL_SYSTEMS),
        "profiles": generation_info.get("profiles", []),
        "cifs": [p.name for p in cifs],
        "generation_mode": "independent stochastic batches across multiple chemical systems and guidance points; not parent-structure refinement",
        "temperature_control": "not supported by MatterGen v1; no fake temperature value was applied",
    }
    write_json(RAW / "mattergen_status.json", STATUS["mattergen"])
    print("\nMatterGen 생성 완료:", len(cifs), "CIF")

    if RUN_OFFICIAL_MATTERGEN_EVALUATION and remaining_seconds() > FINAL_RESERVE_SECONDS + 1800:
        try:
            run_logged(
                ["bash", "-lc",
                 "command -v git-lfs >/dev/null 2>&1 || "
                 "(apt-get update -qq && apt-get install -y -qq git-lfs); "
                 "git lfs install --skip-repo"],
                name="install_git_lfs",
                timeout=1200,
                retries=2,
            )
            reference_rel = "data-release/alex-mp/reference_TRI2024correction.gz"
            reference_path = mg_source / reference_rel
            if not reference_path.exists() or reference_path.stat().st_size < 1_000_000:
                run_logged(
                    ["git", "lfs", "pull", "-I", reference_rel, "--exclude="],
                    name="download_tri2024_reference",
                    cwd=mg_source,
                    timeout=3600,
                    retries=3,
                )

            metrics_path = RAW / "mattergen_metrics.json"
            detailed_path = RAW / "mattergen_detailed_metrics.json"
            relaxed_path = RESULTS / "relaxed_structures.extxyz"
            common_eval = [
                str(mg_py), str(mg_eval_cli),
                f"--structures_path={CIFS}",
                "--relax=True",
                "--structure_matcher=disordered",
                f"--save_as={metrics_path}",
                f"--save_detailed_as={detailed_path}",
                f"--structures_output_path={relaxed_path}",
                "--energy_correction_scheme=TRI2024",
                f"--reference_dataset_path={reference_path}",
            ]
            checkpoint = "MatterSim-v1.0.0-5M.pth" if USE_MATTERSIM_5M else "MatterSim-v1.0.0-1M.pth"
            evaluated_with = checkpoint
            official = run_logged(
                common_eval + [f"--potential_load_path={checkpoint}"],
                name="mattergen_official_evaluation_5m" if USE_MATTERSIM_5M else "mattergen_official_evaluation_1m",
                cwd=mg_source,
                timeout=10800,
                retries=1,
                check=False,
                env_overrides={"MPLBACKEND": "Agg"},
            )
            if official.returncode != 0 and USE_MATTERSIM_5M:
                evaluated_with = "MatterSim-v1.0.0-1M.pth"
                official = run_logged(
                    common_eval + [f"--potential_load_path={evaluated_with}"],
                    name="mattergen_official_evaluation_1m_fallback",
                    cwd=mg_source,
                    timeout=7200,
                    retries=1,
                    check=False,
                    env_overrides={"MPLBACKEND": "Agg"},
                )
            if official.returncode != 0:
                raise RuntimeError("공식 MatterGen 평가가 5M/1M 모두 실패했습니다.")

            STATUS["official_evaluation"].update({
                "success": True,
                "reference": "TRI2024",
                "structure_matcher": "disordered",
                "relaxed": True,
                "potential": evaluated_with,
                "metrics_file": metrics_path.name,
                "detailed_metrics_file": detailed_path.name,
                "relaxed_structures_file": relaxed_path.name,
            })
        except Exception as exc:
            STATUS["official_evaluation"]["error"] = f"{type(exc).__name__}: {exc}"
    write_json(RAW / "official_evaluation_status.json", STATUS["official_evaluation"])


    if RUN_DIRECT_MATTERSIM and remaining_seconds() > FINAL_RESERVE_SECONDS + 1200:
        try:
            ms_py = install_eval_env(
                "mattersim_py312",
                MS["install"]["package"],
                MS["install"]["version"],
            )
            ms_script = WORK / "evaluate_mattersim_candidates.py"
            ms_script.write_text(r"""
import json
import sys
from pathlib import Path

import numpy as np
import torch
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from mattersim.forcefield import MatterSimCalculator

adaptor = AseAtomsAdaptor()
rows = []
try:
    calculator = MatterSimCalculator(
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    calculator = MatterSimCalculator(device="cpu")
    device = "cpu"

for cif in sorted(Path(sys.argv[1]).glob("*.cif")):
    try:
        structure = Structure.from_file(cif)
        atoms = adaptor.get_atoms(structure)
        atoms.calc = calculator
        energy_total = float(atoms.get_potential_energy())
        forces = np.asarray(atoms.get_forces(), dtype=float)
        stress = np.asarray(atoms.get_stress(voigt=False), dtype=float)
        rows.append(
            {
                "cif": cif.name,
                "ok": True,
                "formula": structure.composition.reduced_formula,
                "n_atoms": len(structure),
                "total_energy_eV": energy_total,
                "energy_per_atom_eV": energy_total / max(len(atoms), 1),
                "max_force_eV_per_A": float(
                    np.linalg.norm(forces, axis=1).max()
                ),
                "mean_abs_force_eV_per_A": float(np.abs(forces).mean()),
                "mean_abs_stress_eV_per_A3": float(np.abs(stress).mean()),
                "device": device,
                "model": "MatterSimCalculator managed checkpoint",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "cif": cif.name,
                "ok": False,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

Path(sys.argv[2]).write_text(
    json.dumps(rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(rows, ensure_ascii=False, indent=2))
""", encoding="utf-8")
            run_logged(
                [
                    str(ms_py),
                    str(ms_script),
                    str(CIFS),
                    str(RAW / "mattersim_candidates.json"),
                ],
                name="mattersim_candidate_evaluation",
                timeout=7200,
                retries=2,
                env_overrides={"MPLBACKEND": "Agg"},
            )
            ms_rows = json.loads(
                (RAW / "mattersim_candidates.json").read_text(
                    encoding="utf-8"
                )
            )
            ms_ok = sum(bool(row.get("ok")) for row in ms_rows)
            STATUS["mattersim"].update(
                {
                    "success": bool(ms_rows) and ms_ok == len(ms_rows),
                    "version": MS["install"]["version"],
                    "evaluated": len(ms_rows),
                    "successful_candidates": ms_ok,
                    "role": "per-candidate energy, force, and stress evaluation",
                }
            )
            if ms_ok != len(ms_rows):
                STATUS["mattersim"]["error"] = (
                    f"{ms_ok}/{len(ms_rows)} candidates succeeded"
                )
        except Exception as exc:
            STATUS["mattersim"]["error"] = (
                f"{type(exc).__name__}: {exc}"
            )
    write_json(RAW / "mattersim_status.json", STATUS["mattersim"])

    if RUN_CHGNET_CROSSCHECK and remaining_seconds() > FINAL_RESERVE_SECONDS + 900:
        try:
            chg_py = install_eval_env("chgnet_py312", CHG["install"]["package"], CHG["install"]["version"])
            script = WORK / "evaluate_chgnet_crosscheck.py"
            script.write_text(r'''
import json
import sys
from pathlib import Path

import numpy as np
from pymatgen.core import Structure
from chgnet.model.model import CHGNet

model = CHGNet.load()
rows = []
for cif in sorted(Path(sys.argv[1]).glob("*.cif")):
    try:
        structure = Structure.from_file(cif)
        pred = model.predict_structure(structure)
        energy = float(np.asarray(pred["e"]).reshape(-1)[0])
        forces = np.asarray(pred["f"], dtype=float)
        stress = np.asarray(pred["s"], dtype=float)
        rows.append({
            "cif": cif.name,
            "ok": True,
            "formula": structure.composition.reduced_formula,
            "n_atoms": len(structure),
            "energy_per_atom_eV": energy,
            "max_force_eV_per_A": float(np.linalg.norm(forces, axis=1).max()),
            "mean_abs_stress_GPa": float(np.abs(stress).mean()),
        })
    except Exception as exc:
        rows.append({"cif": cif.name, "ok": False, "error": f"{type(exc).__name__}: {exc}"})
Path(sys.argv[2]).write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(rows, ensure_ascii=False, indent=2))
''', encoding="utf-8")
            run_logged(
                [str(chg_py), str(script), str(CIFS), str(RAW / "chgnet_crosscheck.json")],
                name="chgnet_crosscheck",
                timeout=5400,
                retries=2,
                env_overrides={"MPLBACKEND": "Agg"},
            )
            rows = json.loads((RAW / "chgnet_crosscheck.json").read_text(encoding="utf-8"))
            ok = sum(bool(row.get("ok")) for row in rows)
            STATUS["chgnet"].update({
                "success": ok == len(rows) and bool(rows),
                "version": CHG["install"]["version"],
                "evaluated": len(rows),
                "successful_candidates": ok,
                "role": "independent static energy/force/stress cross-check",
            })
            if ok != len(rows):
                STATUS["chgnet"]["error"] = f"{ok}/{len(rows)} candidates succeeded"
        except Exception as exc:
            STATUS["chgnet"]["error"] = f"{type(exc).__name__}: {exc}"
    write_json(RAW / "chgnet_status.json", STATUS["chgnet"])


    # Build a multi-candidate ranking. Raw energies from different chemical
    # systems are not compared directly. Energy ranks are computed within each
    # chemical system; force/stress ranks are global diagnostics.
    provenance_rows = json.loads(
        (RAW / "candidate_provenance.json").read_text(encoding="utf-8")
    )
    provenance_by_cif = {
        str(row["candidate"]): row for row in provenance_rows
    }

    ms_rows = []
    if (RAW / "mattersim_candidates.json").exists():
        ms_rows = json.loads(
            (RAW / "mattersim_candidates.json").read_text(encoding="utf-8")
        )
    chg_rows = []
    if (RAW / "chgnet_crosscheck.json").exists():
        chg_rows = json.loads(
            (RAW / "chgnet_crosscheck.json").read_text(encoding="utf-8")
        )

    ms_by_cif = {
        str(row.get("cif")): row
        for row in ms_rows
        if row.get("ok")
    }
    chg_by_cif = {
        str(row.get("cif")): row
        for row in chg_rows
        if row.get("ok")
    }

    ranking_rows = []
    for cif in sorted(CIFS.glob("*.cif")):
        provenance = provenance_by_cif.get(cif.name, {})
        ms_row = ms_by_cif.get(cif.name, {})
        chg_row = chg_by_cif.get(cif.name, {})
        ranking_rows.append(
            {
                "candidate": cif.name,
                "formula": ms_row.get("formula") or chg_row.get("formula"),
                "n_atoms": ms_row.get("n_atoms") or chg_row.get("n_atoms"),
                "chemical_system": provenance.get("chemical_system"),
                "profile": provenance.get("profile"),
                "guidance_factor": provenance.get("guidance_factor"),
                "condition_energy_above_hull_eV_per_atom": provenance.get(
                    "energy_above_hull_condition_eV_per_atom"
                ),
                "mattersim_energy_per_atom_eV": ms_row.get(
                    "energy_per_atom_eV"
                ),
                "mattersim_max_force_eV_per_A": ms_row.get(
                    "max_force_eV_per_A"
                ),
                "mattersim_mean_abs_stress_eV_per_A3": ms_row.get(
                    "mean_abs_stress_eV_per_A3"
                ),
                "chgnet_energy_per_atom_eV": chg_row.get(
                    "energy_per_atom_eV"
                ),
                "chgnet_max_force_eV_per_A": chg_row.get(
                    "max_force_eV_per_A"
                ),
                "chgnet_mean_abs_stress_GPa": chg_row.get(
                    "mean_abs_stress_GPa"
                ),
                "both_models_available": bool(ms_row and chg_row),
                "source_sha256": provenance.get("sha256"),
            }
        )

    def finite_number(value):
        return isinstance(value, (int, float)) and math.isfinite(float(value))

    def add_rank(rows, key, output_key, *, groups=None):
        if groups is None:
            groups = {"__all__": rows}
        for group_rows in groups.values():
            valid = [row for row in group_rows if finite_number(row.get(key))]
            valid.sort(key=lambda row: float(row[key]))
            for rank, row in enumerate(valid, 1):
                row[output_key] = rank
            missing_rank = len(valid) + 1
            for row in group_rows:
                row.setdefault(output_key, missing_rank)

    system_groups = {}
    for row in ranking_rows:
        system_groups.setdefault(
            str(row.get("chemical_system") or "unknown"),
            [],
        ).append(row)

    add_rank(
        ranking_rows,
        "mattersim_energy_per_atom_eV",
        "mattersim_energy_rank_within_system",
        groups=system_groups,
    )
    add_rank(
        ranking_rows,
        "chgnet_energy_per_atom_eV",
        "chgnet_energy_rank_within_system",
        groups=system_groups,
    )
    add_rank(
        ranking_rows,
        "mattersim_max_force_eV_per_A",
        "mattersim_force_rank_global",
    )
    add_rank(
        ranking_rows,
        "chgnet_max_force_eV_per_A",
        "chgnet_force_rank_global",
    )
    add_rank(
        ranking_rows,
        "mattersim_mean_abs_stress_eV_per_A3",
        "mattersim_stress_rank_global",
    )
    add_rank(
        ranking_rows,
        "chgnet_mean_abs_stress_GPa",
        "chgnet_stress_rank_global",
    )

    for row in ranking_rows:
        rank_keys = [
            "mattersim_energy_rank_within_system",
            "chgnet_energy_rank_within_system",
            "mattersim_force_rank_global",
            "chgnet_force_rank_global",
            "mattersim_stress_rank_global",
            "chgnet_stress_rank_global",
        ]
        row["reciprocal_rank_score"] = sum(
            1.0 / (60.0 + float(row[key])) for key in rank_keys
        )
        ms_e = row.get("mattersim_energy_per_atom_eV")
        chg_e = row.get("chgnet_energy_per_atom_eV")
        row["energy_model_gap_eV_per_atom"] = (
            abs(float(ms_e) - float(chg_e))
            if finite_number(ms_e) and finite_number(chg_e)
            else None
        )
        row["ranking_warning"] = (
            "Raw energy values are model-specific. Cross-system energy "
            "magnitudes are not directly compared; energy ranks are computed "
            "within each chemical system."
        )

    ranking_rows.sort(
        key=lambda row: (
            -float(row["reciprocal_rank_score"]),
            str(row["candidate"]),
        )
    )
    for index, row in enumerate(ranking_rows, 1):
        row["rank"] = index

    ranking_json = RAW / "ranked_candidates.json"
    write_json(ranking_json, ranking_rows)
    ranking_csv = RESULTS / "ranked_candidates.csv"
    if ranking_rows:
        fieldnames = list(ranking_rows[0].keys())
        with ranking_csv.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(ranking_rows)

    STATUS["ranked_candidate_count"] = len(ranking_rows)
    STATUS["ranking_file"] = str(ranking_csv.relative_to(WORK))
    print("\n===== TOP RANKED CANDIDATES =====")
    print(
        json.dumps(
            ranking_rows[: min(30, len(ranking_rows))],
            ensure_ascii=False,
            indent=2,
        )
    )

    STATUS["scientific_boundary"] = (
        "Candidates are stochastic MatterGen outputs. MatterSim/CHGNet are ML force fields; "
        "top candidates still require DFT relaxation, phonon/dynamic checks, and experimental validation."
    )
    FINAL_ZIP = package_all("completed")

except BaseException as exc:
    fatal_error = f"{type(exc).__name__}: {exc}"
    STATUS["traceback"] = traceback.format_exc()
    print("\n실행 오류:", fatal_error)
    print(STATUS["traceback"])
    FINAL_ZIP = package_all("failed_but_packaged", fatal_error)

finally:
    if FINAL_ZIP is None or not FINAL_ZIP.exists():
        FINAL_ZIP = package_all("interrupted_but_packaged", fatal_error)
    print("\n최종 ZIP:", FINAL_ZIP)
    print("최종 상태:", STATUS.get("status"))
    if DRIVE_OUT is not None:
        print("Drive 복사:", DRIVE_OUT / FINAL_ZIP.name)
    if BROWSER_DOWNLOAD_FINAL:
        try:
            from google.colab import files
            files.download(str(FINAL_ZIP))
        except Exception as download_exc:
            print("브라우저 다운로드 호출 실패:", repr(download_exc))
            print("Colab 파일 패널에서 받으세요:", FINAL_ZIP)


In [ ]:
#@title 3. Display the final multi-candidate ranking
from pathlib import Path

import pandas as pd
from IPython.display import display

ranking_path = Path(
    "/content/direct_real_models_v11_prompt_rag/results/ranked_candidates.csv"
)
if ranking_path.exists():
    ranked = pd.read_csv(ranking_path)
    display(ranked.head(120))
else:
    print(
        "The ranking table was not created. Check the downloaded ZIP for "
        "run_status.json and complete logs."
    )


## Optional genomic variant evaluation (AlphaGenome)

This is an intermediate expert-evaluation branch, not a sequence generator. It evaluates a batch of candidate variants, preserves per-candidate uncertainty and model disagreement, and returns a ranked table. The API key is requested at runtime and is never written to the notebook or artifacts. AlphaGenome API use is subject to Google's non-commercial terms; outputs are research-only and not clinical evidence.

In [ ]:
#@title 4. Evaluate genomic candidates with AlphaGenome (run separately from MatterGen)
import getpass
import json
import os
import sys

# Environment configuration can enable the branch without editing the notebook.
if GENOMIC_MODE == 'OFF' and os.environ.get('ALPHAGENOME_API_KEY', '').strip() and GENOMIC_CANDIDATES_JSON.strip():
    GENOMIC_MODE = 'ALPHAGENOME_EVALUATION_ONLY'
if GENOMIC_MODE == 'OFF':
    print('AlphaGenome branch is OFF; material pipeline is unchanged.')
else:
    ALPHAGENOME_API_KEY = os.environ.get('ALPHAGENOME_API_KEY', '').strip()
    if not ALPHAGENOME_API_KEY:
        ALPHAGENOME_API_KEY = getpass.getpass('AlphaGenome API key (hidden; never stored): ').strip()
    if not ALPHAGENOME_API_KEY:
        raise ValueError('An AlphaGenome API key is required when GENOMIC_MODE is enabled.')
    os.environ['ALPHAGENOME_API_KEY'] = ALPHAGENOME_API_KEY
    if not GENOMIC_CANDIDATES_JSON.strip():
        raise ValueError('Set GENOMIC_CANDIDATES_JSON to a JSON file or inline JSON list when genomic mode is enabled.')
    candidate_source = Path(GENOMIC_CANDIDATES_JSON)
    candidate_rows = json.loads(candidate_source.read_text(encoding='utf-8')) if candidate_source.exists() else json.loads(GENOMIC_CANDIDATES_JSON)
    if not isinstance(candidate_rows, list) or not candidate_rows:
        raise ValueError('Genomic candidates must be a non-empty JSON list.')

    # Install the official client only for this explicitly enabled branch.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'alphagenome @ git+https://github.com/google-deepmind/alphagenome.git'], check=True)

    from discovery_os.alphagenome import GenomicEvaluationPipeline, VariantCandidate
    candidates = [VariantCandidate(**row) for row in candidate_rows]
    ranked_genomic_candidates = GenomicEvaluationPipeline().run(candidates)
    display(pd.DataFrame(ranked_genomic_candidates))
    with open('/content/alphagenome_ranked_candidates.json', 'w', encoding='utf-8') as handle:
        json.dump(ranked_genomic_candidates, handle, ensure_ascii=False, indent=2)
    print('Saved: /content/alphagenome_ranked_candidates.json')
